# GKX (2019) — Empirical Asset Pricing via Machine Learning
**Author:** Francesco Pio De Girolamo, Brice Namy &nbsp;·&nbsp; IEOR E4733 &nbsp;·&nbsp; Spring 2026

End-to-end Colab notebook for the two pipelines:

1. **`paper`** — Gu, Kelly & Xiu (2020) replication, 1957–2016, gross of TC.
2. **`improved`** — extended to 2024, net of impact-aware (FIM-style) TC.

## For the grader

1. Zip the entire project as `empirical_asset_pricing_ml.zip`.
2. In Google Drive, create a folder named exactly **`Algorithmic Trading Project`** and upload the zip there.
3. Open this notebook in Colab and run all cells. Section 0 mounts Drive and unzips the project.
4. Section 7 runs both variants back-to-back. End-to-end ~2–3 hrs per variant on a Colab Pro T4.

WRDS credentials are required for the data-fetch step in Section 3.

# Section 0 — First-time setup

### 0.1 Mount Google Drive & extract project

In [ ]:
# Mount Drive and extract the project zip (FIRST TIME ONLY).
# Expected Drive layout:
#   /content/drive/MyDrive/Algorithmic Trading Project/
#     ├── empirical_asset_pricing_ml.zip      ← this project
#     └── PredictorData2023.xlsx              ← Welch-Goyal macro data
import os, zipfile, shutil
from google.colab import drive

drive.mount('/content/drive')

DRIVE_BASE = '/content/drive/MyDrive/Algorithmic Trading Project'
ZIP_CANDIDATES = [
    f'{DRIVE_BASE}/empirical_asset_pricing_ml.zip',
]
PROJECT = '/content/empirical_asset_pricing_ml'

ZIP_PATH = next((p for p in ZIP_CANDIDATES if os.path.exists(p)), None)
if ZIP_PATH is None:
    raise FileNotFoundError(
        "No project zip found. Upload 'empirical_asset_pricing_ml.zip' to:\n"
        f'  {DRIVE_BASE}/'
    )

print('Using project zip:', ZIP_PATH)

# Re-extract only if the project is not already present.
# To force a clean extraction, uncomment the shutil.rmtree line below.
# if os.path.exists(PROJECT): shutil.rmtree(PROJECT)

if not os.path.exists(PROJECT):
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall('/content')
    print(f'✅ Extracted project to {PROJECT}')
else:
    print(f'ℹ️  Project already extracted at {PROJECT}')

### 0.2 Install dependencies

In [ ]:
%cd /content/empirical_asset_pricing_ml
!pip install -r requirements.txt --quiet
!pip install wrds --no-deps --quiet
# Pin numpy/pandas/pyarrow to versions used for the verified runs.
!pip install --force-reinstall --no-cache-dir "numpy==1.26.4" "pandas==2.2.2" "pyarrow==15.0.2" --quiet

# Section 1 — Post-restart setup

After Colab restarts (e.g. for `--force-reinstall` to take effect), run Section 1 to re-establish paths, GPU access, cache restoration, and the backup helper.

### 1.1 Python path & working directory

In [ ]:
import sys, os, pathlib
PROJECT = '/content/empirical_asset_pricing_ml'
if PROJECT not in sys.path:
    sys.path.insert(0, PROJECT)
os.chdir(PROJECT)
print('cwd:', pathlib.Path.cwd())

### 1.2 GPU and RAM check

In [ ]:
import torch, psutil
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
print(f'RAM total: {psutil.virtual_memory().total / 1e9:.1f} GB')

### 1.3 Restore cache + results from Drive

If you have already run the pipeline once and backed up cache / outputs to Drive, restore them here so you don't have to refetch from WRDS.

In [ ]:
import shutil, pathlib
DRIVE_BASE   = pathlib.Path('/content/drive/MyDrive/Algorithmic Trading Project')
CACHE_BACKUP = DRIVE_BASE / 'cache_backup'
OUTPUTS_BACKUP = DRIVE_BASE / 'outputs_backup'
CACHE_LOCAL  = pathlib.Path('data/cache')
OUTPUTS_LOCAL  = pathlib.Path('outputs')

for src, dst in [(CACHE_BACKUP, CACHE_LOCAL), (OUTPUTS_BACKUP, OUTPUTS_LOCAL)]:
    if src.exists():
        dst.mkdir(parents=True, exist_ok=True)
        for item in src.iterdir():
            target = dst / item.name
            if item.is_dir():
                shutil.copytree(item, target, dirs_exist_ok=True)
            else:
                shutil.copy2(item, target)
        print(f'  ✅ Restored {src} → {dst}')
    else:
        print(f'  ⏭️  {src} (not present, skipping)')

### 1.4 Patch `all_models.py` — enable GPU for neural networks

In [ ]:
import torch
if torch.cuda.is_available():
    # Confirm the NN training code will pick up CUDA.
    # The training functions in src/models/all_models.py read DEVICE = 'cuda' when available.
    from src.models import all_models as am
    if hasattr(am, 'DEVICE'):
        am.DEVICE = 'cuda'
        print('  ✅ all_models.DEVICE = cuda')
    else:
        print('  ℹ️  all_models has no DEVICE constant; NNs use torch.cuda.is_available() inline.')
else:
    print('  ⚠️  No GPU available — neural-net training will fall back to CPU and be slow.')

### 1.5 Backup helper

After every long-running step we sync `data/cache` and `outputs` back to Drive so a Colab disconnect doesn't lose hours of compute.

In [ ]:
import os, shutil
from datetime import datetime

def backup_to_drive(label: str) -> None:
    '''Copy data/cache and outputs to Drive (overwrite-only).'''
    DRIVE = '/content/drive/MyDrive/Algorithmic Trading Project'
    PROJECT = '/content/empirical_asset_pricing_ml'
    pairs = [
        (f'{PROJECT}/data/cache', f'{DRIVE}/cache_backup'),
        (f'{PROJECT}/outputs',    f'{DRIVE}/outputs_backup'),
        (f'{PROJECT}/logs',       f'{DRIVE}/logs_backup'),
    ]
    for src, dst in pairs:
        if os.path.exists(src):
            os.makedirs(dst, exist_ok=True)
            shutil.copytree(src, dst, dirs_exist_ok=True)
    stamp = datetime.now().strftime('%H:%M:%S')
    print(f'  [{stamp}] backup: {label}')

# Section 2 — Choose variant

Run Sections 3–5 with one variant, then change `VARIANT` and re-run Sections 3–5 for the other. Or skip to **Section 7** to run both back-to-back.

In [ ]:
# ⚠️ CHANGE THIS to switch pipelines.
# Available variants: 'paper' (1957-2016, no TC) or 'improved' (1957-2024, impact-aware TC)

VARIANTS = ['paper', 'improved']

VARIANT = 'improved'          # choose one from VARIANTS
WRDS_USERNAME = 'your_wrds_user'  # set to your WRDS username

print('Variant:', VARIANT)
print('Available variants:', VARIANTS)

# Section 3 — Data pipeline

Each `--data-step` is cacheable independently. Re-run a step only if you want to bust its cache; otherwise it skips with a short message.

### 3.1 Fetch raw data (WRDS + Welch-Goyal macro)

In [ ]:
!python main.py --mode data-only --data-step fetch --variant $VARIANT --wrds-username $WRDS_USERNAME

In [ ]:
# (Same thing using Python interpolation, in case the shell-arg version misbehaves.)
import subprocess
subprocess.run(
    ['python', 'main.py', '--mode', 'data-only', '--data-step', 'fetch',
     '--variant', VARIANT, '--wrds-username', WRDS_USERNAME],
    check=True,
)

### 3.2 Merge CRSP + Compustat

In [ ]:
!python main.py --mode data-only --data-step merge --variant {VARIANT}

### 3.3 Build characteristics

In [ ]:
!python main.py --mode data-only --data-step chars --variant {VARIANT}

### 3.4 Build feature matrix (Kronecker product + macro interactions + SIC dummies)

In [ ]:
!python main.py --mode data-only --data-step features --variant {VARIANT}

### 3.5 Backup data cache to Drive

In [ ]:
backup_to_drive(f'data_{VARIANT}')

# Section 4 — Training

10 base models: 5 linear-style (CPU, fast), 1 tree ensemble, 4 neural nets (GPU). Each `train --models X` writes `outputs/<variant>/models/<X>.pkl` (predictions, not fitted objects).

Models trained: **OLS-3, ENet+H, PCR, PLS, GLM+H, GBRT+H, NN1, NN2, NN3, NN4**.
Ensembles **ENS-AVG** and **ENS-MSE** are built automatically during `--mode evaluate`.

## Stage 4a: Linear / penalised models (CPU, fast)

In [ ]:
for m in ['OLS-3', 'ENet+H', 'PCR', 'PLS', 'GLM+H']:
    !python main.py --mode train --variant $VARIANT --models {m}
    backup_to_drive(f'linear_{VARIANT}')

## Stage 4b: Tree ensemble (CPU, moderate)

GBRT+H uses sklearn `HistGradientBoostingRegressor` with staged-prediction tuning.

In [ ]:
for m in ['GBRT+H']:
    !python main.py --mode train --variant $VARIANT --models {m}
    backup_to_drive(f'trees_{VARIANT}')

## Stage 4c: Neural networks (GPU, slow)

Each NN<k> trains a 10-seed ensemble. With T4 GPU, ~10–25 min per depth for the paper variant, ~15–35 min for improved.

In [ ]:
for m in ['NN1', 'NN2', 'NN3', 'NN4']:
    !python main.py --mode train --variant $VARIANT --models {m}
    backup_to_drive(f'{m}_{VARIANT}')

# Section 5 — Evaluate + Variable Importance

Evaluate also constructs ENS-AVG and ENS-MSE from the trained base models. Regimes slices each model's H-L series by NBER / VIX / decade. Importance computes Δ R² when each feature is zeroed.

### 5.1 Evaluate

In [ ]:
!python main.py --mode evaluate --variant {VARIANT}

### 5.2 Inspect comprehensive metrics

In [ ]:
import pandas as pd
cm = pd.read_csv(f'outputs/{VARIANT}/comprehensive.csv')
print(f'Comprehensive metrics — {VARIANT}')
display(cm.round(4))

### 5.3 Inspect Diebold-Mariano matrix

In [ ]:
import pandas as pd
dm = pd.read_csv(f'outputs/{VARIANT}/dm_table.csv', index_col=0)
pv = pd.read_csv(f'outputs/{VARIANT}/dm_pvalues.csv', index_col=0)
print('DM statistics')
display(dm.round(3))
print('DM p-values')
display(pv.round(3))

### 5.4 Regime-conditional evaluation

NBER recessions (peak-to-trough), VIX terciles, calendar decades.

In [ ]:
!python main.py --mode regimes --variant {VARIANT}

In [ ]:
import pandas as pd
rg = pd.read_csv(f'outputs/{VARIANT}/regimes.csv')
# Show NBER recession vs expansion side-by-side
rec = rg[rg.regime_kind == 'nber'].pivot(index='model', columns='regime',
                                          values=['sharpe_net', 'sharpe_gross'])
display(rec.round(3))
# And VIX terciles
vix = rg[rg.regime_kind == 'vix'].pivot(index='model', columns='regime',
                                         values='sharpe_net')
display(vix.round(3))

### 5.5 Variable importance

Zero-set Δ R² per characteristic; aggregates the 920-feature Kronecker expansion back to 94 base characteristics. Slow — restrict to a subset of models if needed.

In [ ]:
# Pick which models to compute importance for (subset is fine — this is slow for some models).
IMPORTANCE_MODELS = ['OLS-3', 'ENet+H', 'PCR', 'PLS', 'GBRT+H', 'NN3', 'NN4']

import subprocess
subprocess.run([
    'python', 'main.py', '--mode', 'importance', '--variant', VARIANT,
    '--models', *IMPORTANCE_MODELS,
], check=False)
backup_to_drive(f'importance_{VARIANT}')

### 5.6 Inspect variable importance

In [ ]:
import pandas as pd
vi = pd.read_csv(f'outputs/{VARIANT}/var_importance.csv', index_col=0)
vi['mean'] = vi.mean(axis=1)
print(f'Top-15 characteristics by mean importance — {VARIANT}')
display(vi.sort_values('mean', ascending=False).head(15).round(3))

# Section 6 — Dashboard

The Streamlit dashboard reads every `outputs/<variant>/` directory and surfaces all metrics, charts, and regime slices in one place. Locally just `streamlit run src/dashboard/app.py`; in Colab we tunnel via ngrok.

In [ ]:
# In Colab, expose Streamlit via ngrok. You need a free ngrok token from
# https://dashboard.ngrok.com/get-started/your-authtoken
!pip install -q streamlit pyngrok
NGROK_AUTHTOKEN = ''  # paste your ngrok token here

In [ ]:
from pyngrok import ngrok, conf
import time, subprocess

if NGROK_AUTHTOKEN:
    conf.get_default().auth_token = NGROK_AUTHTOKEN
# Kill any old tunnels
ngrok.kill()
# Public URL for Streamlit's default port 8501
public_url = ngrok.connect(8501, 'http')
print('Public dashboard URL:', public_url)

In [ ]:
# Launch (Colab helper). Adapt for your environment if needed.
import subprocess, sys
proc = subprocess.Popen(
    [sys.executable, '-m', 'streamlit', 'run', 'src/dashboard/app.py',
     '--server.port', '8501',
     '--server.headless', 'true',
     '--browser.gatherUsageStats', 'false'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
print('Streamlit PID:', proc.pid)
print('Open the ngrok URL printed in the previous cell.')
print('Use `proc.terminate()` to stop the dashboard.')

# Section 7 — Convenience: run both variants end-to-end

Runs `paper` followed by `improved` with the full pipeline (data → train → evaluate → regimes → importance). Each variant writes to its own `outputs/<variant>/` directory.

In [ ]:
import subprocess

BASE_MODELS = ['OLS-3', 'ENet+H', 'PCR', 'PLS', 'GLM+H',
               'GBRT+H',
               'NN1', 'NN2', 'NN3', 'NN4']

for v in VARIANTS:
    print(f'\n========== Running variant: {v} ==========')
    subprocess.run(['python', 'main.py', '--mode', 'data-only', '--variant', v,
                    '--wrds-username', WRDS_USERNAME], check=True)
    for m in BASE_MODELS:
        subprocess.run(['python', 'main.py', '--mode', 'train', '--variant', v,
                        '--models', m], check=False)
    subprocess.run(['python', 'main.py', '--mode', 'evaluate',   '--variant', v], check=True)
    subprocess.run(['python', 'main.py', '--mode', 'regimes',    '--variant', v], check=False)
    subprocess.run(['python', 'main.py', '--mode', 'importance', '--variant', v,
                    '--models', 'OLS-3', 'ENet+H', 'PCR', 'PLS', 'GBRT+H', 'NN3', 'NN4'],
                   check=False)
    backup_to_drive(f'all_{v}')

# Section 8 — Final backup of code + outputs

Save the produced `outputs/`, `logs/`, and a snapshot of the code back to Drive so you can re-mount in a fresh runtime later without re-fetching from WRDS.

In [ ]:
import pathlib
DRIVE_BASE   = pathlib.Path('/content/drive/MyDrive/Algorithmic Trading Project')
CACHE_BACKUP = DRIVE_BASE / 'cache_backup'
CACHE_LOCAL  = pathlib.Path('data/cache')
OUTPUTS_BACKUP = DRIVE_BASE / 'outputs_backup'
OUTPUTS_LOCAL  = pathlib.Path('outputs')

In [ ]:
import os, shutil
backup_dir = str(DRIVE_BASE / 'outputs_backup')
os.makedirs(backup_dir, exist_ok=True)
for folder in ['outputs', 'logs']:
    src = f'/content/empirical_asset_pricing_ml/{folder}'
    if os.path.exists(src):
        dst = os.path.join(backup_dir, folder)
        if os.path.exists(dst):
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
print('✅ Outputs & logs backed up to:', backup_dir)

In [ ]:
import subprocess
from datetime import datetime
stamp    = datetime.now().strftime('%Y%m%d_%H%M%S')
zip_name = f'empirical_asset_pricing_ml_code_{stamp}.zip'
zip_path = str(DRIVE_BASE / zip_name)
subprocess.run([
    'zip', '-r', zip_path, 'empirical_asset_pricing_ml',
    '--exclude', '*/data/cache/*',
    '--exclude', '*/data/gwz_extracted/*',
    '--exclude', '*/__pycache__/*',
    '--exclude', '*/.git/*',
    '--exclude', '*/outputs/*',
    '--exclude', '*/logs/*',
], cwd='/content', check=True)
print('✅ Code snapshot saved at:', zip_path)

In [ ]:
# ============================================================
#  Save everything to Drive — cache + outputs, overwrite-only
# ============================================================
import os, shutil
from datetime import datetime

DRIVE_BASE_STR = '/content/drive/MyDrive/Algorithmic Trading Project'
PROJECT = '/content/empirical_asset_pricing_ml'

pairs = [
    (f'{PROJECT}/data/cache', f'{DRIVE_BASE_STR}/cache_backup'),
    (f'{PROJECT}/outputs',    f'{DRIVE_BASE_STR}/outputs_backup'),
    (f'{PROJECT}/logs',       f'{DRIVE_BASE_STR}/logs_backup'),
]

for src, dst in pairs:
    if not os.path.exists(src):
        print(f'  ⏭️  {src} (not present, skipping)')
        continue
    os.makedirs(dst, exist_ok=True)
    shutil.copytree(src, dst, dirs_exist_ok=True)
    n_files = sum(len(f) for _, _, f in os.walk(dst))
    size_mb = sum(
        os.path.getsize(os.path.join(d, f))
        for d, _, fs in os.walk(dst) for f in fs
    ) / 1e6
    print(f'  ✅ {src} → {dst}  ({n_files} files, {size_mb:.1f} MB)')

stamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
print(f'\n[{stamp}] Backup complete')

# Appendix — Variant map

| Variant     | Sample period | TC model              | Test window  |
|-------------|---------------|-----------------------|--------------|
| `paper`     | 1957 – 2016   | 0 bps (gross)         | 1987 – 2016 |
| `improved`  | 1957 – 2024   | Impact-aware FIM-style | 1987 – 2024 |

Each variant writes to its own `outputs/<variant>/` directory and uses its own cached feature matrix at `data/cache/feature_matrix_<variant>.parquet`.

**Outputs produced per variant:**

* `comprehensive.csv` — 12-model risk-adjusted metrics table
* `oos_r2.csv` — pooled monthly OOS R² per model
* `sharpe_table.csv` — TC-sensitivity grid
* `dm_table.csv` & `dm_pvalues.csv` — Diebold-Mariano pairwise tests
* `portfolio_returns.pkl` — per-decile and H-L returns, gross + net, plus turnover series
* `regimes.csv` — NBER × VIX × decade Sharpes per model
* `var_importance.csv` — chars × models matrix of Δ R²
* `metrics.json` — full diagnostic bundle (training stats, ensemble weights, etc.)
* `models/*.pkl` — per-model prediction dataframes (not fitted objects)